# X-CGM-JEPA — minimal Colab trainer

Self-contained. Upload your CSV to Google Drive, set `DATA_PATH` below, run top to bottom.

`L_total = L_CGM + w * L_Glu`. CGM target = EMA encoder (stop-grad); glucodensity
target = trainable teacher (gets gradients). Lightning trainer, val loss every
epoch, EarlyStopping(patience=3).

In [ ]:
!pip -q install pytorch-lightning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- config ----
DRIVE_DIR = "/content/drive/MyDrive/XJepa"                 # your Drive folder
TRAIN_NPZ = f"{DRIVE_DIR}/x_jepa_paired_train.npz"          # precomputed pairs (windows + images)
VAL_NPZ   = f"{DRIVE_DIR}/x_jepa_paired_val.npz"

WINDOW      = 128     # glucose window length
PATCH       = 8       # CGM patch size  -> 16 CGM patches
GRIDSIZE    = 32      # KDE grid size   -> 16 glucodensity patches (8x8 tiles)

EMBED_DIM         = 96
GLUCO_LOSS_WEIGHT = 1.0
SIGREG_WEIGHT     = 1.0   # anti-collapse on glucodensity only; 0 = released behaviour
LR                = 1e-3
N_CGM_TARGETS     = 4     # masked CGM blocks
N_GLU_TARGETS     = 8     # masked glucodensity patches
EMA_BASE          = 0.999
BATCH_SIZE        = 256
MAX_EPOCHS        = 50
PATIENCE          = 3
SEED              = 0

N_CGM_PATCHES = WINDOW // PATCH
N_GLU_PATCHES = (GRIDSIZE // 8) ** 2

In [ ]:
import copy, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset

## Load precomputed paired (glucose window, glucodensity image) arrays

In [ ]:
d = np.load(TRAIN_NPZ); train_w, train_i = d["windows"], d["images"]
d = np.load(VAL_NPZ);   val_w,   val_i   = d["windows"], d["images"]
print("train pairs:", len(train_w), train_w.shape, train_i.shape)
print("val   pairs:", len(val_w),   val_w.shape,   val_i.shape)

class PairedDataset(Dataset):
    def __init__(self, w, i):
        self.w = torch.from_numpy(w).float(); self.i = torch.from_numpy(i).float()
    def __len__(self): return len(self.w)
    def __getitem__(self, k): return self.w[k], self.i[k]

## Model pieces (inlined from the repo)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, : x.size(1)]

class JepaBlock(nn.Module):
    def __init__(self, dim, n_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout))
    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

def _sinusoidal_table(n_positions, dim):
    pe = torch.zeros(n_positions, dim)
    pos = torch.arange(0, n_positions, dtype=torch.float).unsqueeze(1)
    div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
    return pe

class JepaEncoder(nn.Module):
    """CGM encoder: glucose (B, T) -> (B, n_patches, embed_dim). Per-window z-score."""
    def __init__(self, n_time_steps, patch_size=8, embed_dim=96, n_layers=3, n_heads=6,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.n_patches = n_time_steps // patch_size
        self.patch_embed = nn.Conv1d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_enc = PositionalEncoding(embed_dim, max_len=self.n_patches)
        self.blocks = nn.ModuleList([JepaBlock(embed_dim, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, glucose, keep=None):
        mean = glucose.mean(dim=1, keepdim=True)
        std = glucose.std(dim=1, keepdim=True).clamp_min(1e-6)
        glucose = (glucose - mean) / std
        x = self.pos_enc(self.patch_embed(glucose.unsqueeze(1)).transpose(1, 2))
        if keep is not None:
            x = x.gather(1, keep.unsqueeze(-1).expand(-1, -1, x.size(-1)))
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

class JepaPredictor(nn.Module):
    """P_CGM: predict masked CGM latents from context latents + target positions."""
    def __init__(self, embed_dim, n_patches, pred_dim=None, n_layers=2, n_heads=4,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        pred_dim = pred_dim or max(embed_dim // 2, n_heads)
        self.in_proj = nn.Linear(embed_dim, pred_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pred_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.register_buffer("pos_table", _sinusoidal_table(n_patches, pred_dim))
        self.blocks = nn.ModuleList([JepaBlock(pred_dim, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(pred_dim)
        self.out_proj = nn.Linear(pred_dim, embed_dim)
    def forward(self, context, context_idx, target_idx):
        b, n_ctx, _ = context.shape
        ctx = self.in_proj(context) + self.pos_table[context_idx].unsqueeze(0)
        tgt = self.mask_token.expand(b, target_idx.numel(), -1) + self.pos_table[target_idx].unsqueeze(0)
        x = torch.cat([ctx, tgt], dim=1)
        for blk in self.blocks:
            x = blk(x)
        return self.out_proj(self.norm(x[:, n_ctx:, :]))

class GlucoEncoder(nn.Module):
    """Glucodensity image (B, 32, 32, 3) -> (B, 16, embed_dim)."""
    def __init__(self, gridsize=32, patch=8, in_ch=3, embed_dim=96, n_layers=3, n_heads=6,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.patch = patch
        self.n_patches = (gridsize // patch) ** 2
        self.embed = nn.Linear(patch * patch * in_ch, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim, max_len=self.n_patches)
        self.blocks = nn.ModuleList([JepaBlock(embed_dim, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
    def patchify(self, img):
        b, h, w, c = img.shape; p = self.patch
        x = img.reshape(b, h // p, p, w // p, p, c).permute(0, 1, 3, 2, 4, 5)
        return x.reshape(b, (h // p) * (w // p), p * p * c)
    def forward(self, img, keep=None):
        x = self.pos_enc(self.embed(self.patchify(img)))
        if keep is not None:
            x = x.gather(1, keep.unsqueeze(-1).expand(-1, -1, x.size(-1)))
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

class GluPredictor(nn.Module):
    """P_Glu: predict masked glucodensity latents from the CGM context (cross-modal)."""
    def __init__(self, num_gluco_patches=16, embed_dim=96, pred_dim=48, n_layers=1, n_heads=2,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.num_gluco_patches = num_gluco_patches
        self.predictor_embed = nn.Linear(embed_dim, pred_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pred_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.register_buffer("pos", _sinusoidal_table(num_gluco_patches, pred_dim))
        self.blocks = nn.ModuleList([JepaBlock(pred_dim, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(pred_dim)
        self.predictor_proj = nn.Linear(pred_dim, embed_dim)
    def forward(self, cgm_context, gluco_masks=None):
        b, lc, _ = cgm_context.shape
        x = self.predictor_embed(cgm_context)
        blanks = self.mask_token.expand(b, self.num_gluco_patches, -1) + self.pos.unsqueeze(0)
        x = torch.cat([x, blanks], dim=1)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)[:, lc:]
        if gluco_masks is not None:
            idx = gluco_masks.unsqueeze(-1).expand(-1, -1, x.size(-1))
            x = x.gather(1, idx)
        return self.predictor_proj(x)

## Masking, EMA, dual loss, collapse metric

In [ ]:
def sample_block_mask(n_patches, n_targets, min_block, max_block, rng, max_attempts=100):
    for _ in range(max_attempts):
        occupied = set(); placed = 0
        for _ in range(n_targets):
            size = rng.randint(min_block, max_block)
            for _ in range(20):
                start = rng.randint(0, n_patches - size)
                block = set(range(start, start + size))
                if not (block & occupied):
                    occupied |= block; placed += 1; break
        context = [i for i in range(n_patches) if i not in occupied]
        if placed == n_targets and context:
            return context, sorted(occupied)
    raise RuntimeError("could not place mask blocks")

@torch.no_grad()
def ema_update(target, online, m):
    for pt, po in zip(target.parameters(), online.parameters()):
        pt.mul_(m).add_(po.detach(), alpha=1.0 - m)
    for bt, bo in zip(target.buffers(), online.buffers()):
        bt.copy_(bo)

def momentum_at(step, total, base, final=1.0):
    if total <= 1: return final
    p = min(step / (total - 1), 1.0)
    return final - (final - base) * (math.cos(math.pi * p) + 1) / 2

@torch.no_grad()
def collapse_metrics(latents):
    z = latents.reshape(-1, latents.size(-1)).float()
    std = z.std(dim=0).mean().item()
    zc = z - z.mean(dim=0, keepdim=True)
    cov = (zc.T @ zc) / max(z.size(0) - 1, 1)
    ev = torch.linalg.eigvalsh(cov).clamp_min(0)
    denom = (ev ** 2).sum()
    eff_rank = (ev.sum() ** 2 / denom).item() if denom > 0 else 0.0
    return std, eff_rank

def sigreg(z, num_slices=256, k=17):
    # LeJEPA SIGReg: push embeddings toward isotropic N(0,1) so they can't collapse.
    d = z.size(1)
    a = torch.randn(d, num_slices, device=z.device, dtype=z.dtype)
    a = a / a.norm(dim=0, keepdim=True)
    proj = z @ a
    t = torch.linspace(-5, 5, k, device=z.device, dtype=z.dtype)
    phi = torch.exp(-0.5 * t ** 2)                        # N(0,1) characteristic function
    xt = proj.unsqueeze(-1) * t
    re = torch.cos(xt).mean(0); im = torch.sin(xt).mean(0)  # empirical CF (cos/sin, MPS/GPU-safe)
    return torch.trapz(((re - phi) ** 2 + im ** 2) * phi, t, dim=1).mean()

def x_forward_loss(cgm_encoder, cgm_encoder_ema, cgm_predictor, glu_encoder, glu_predictor,
                   glucose, gluco_img, cgm_ctx_idx, cgm_tgt_idx, gluco_tgt_idx,
                   gluco_loss_weight=1.0, sigreg_weight=0.0):
    b = glucose.size(0)
    with torch.no_grad():
        cgm_full = cgm_encoder_ema(glucose)
        cgm_full = F.layer_norm(cgm_full, (cgm_full.size(-1),))
        cgm_targets = cgm_full[:, cgm_tgt_idx, :]
    keep = cgm_ctx_idx.unsqueeze(0).expand(b, -1)
    cgm_context = cgm_encoder(glucose, keep=keep)
    cgm_pred = cgm_predictor(cgm_context, cgm_ctx_idx, cgm_tgt_idx)
    cgm_loss = F.l1_loss(cgm_pred, cgm_targets)
    glu_full = glu_encoder(gluco_img)
    reg = sigreg(glu_full.reshape(-1, glu_full.size(-1))) if sigreg_weight > 0 else glu_full.new_zeros(())
    glu_full = F.layer_norm(glu_full, (glu_full.size(-1),))
    glu_targets = glu_full[:, gluco_tgt_idx, :]
    gluco_masks = gluco_tgt_idx.unsqueeze(0).expand(b, -1)
    glu_pred = glu_predictor(cgm_context, gluco_masks)
    gluco_loss = F.l1_loss(glu_pred, glu_targets)
    total = cgm_loss + gluco_loss_weight * gluco_loss + sigreg_weight * reg
    return total, cgm_loss.detach(), gluco_loss.detach(), reg.detach()

## LightningModule + training

In [ ]:
class XJepa(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.cgm_encoder = JepaEncoder(WINDOW, PATCH, EMBED_DIM, 3, 6)
        self.cgm_encoder_ema = copy.deepcopy(self.cgm_encoder)
        for p in self.cgm_encoder_ema.parameters():
            p.requires_grad = False
        self.cgm_predictor = JepaPredictor(EMBED_DIM, N_CGM_PATCHES)
        self.glu_encoder = GlucoEncoder(GRIDSIZE, 8, 3, EMBED_DIM)
        self.glu_predictor = GluPredictor(N_GLU_PATCHES, EMBED_DIM)
        self.rng = random.Random(SEED)
        # Fixed masks so val loss is comparable across epochs.
        self.val_ctx = list(range(N_CGM_PATCHES - N_CGM_TARGETS))
        self.val_tgt = list(range(N_CGM_PATCHES - N_CGM_TARGETS, N_CGM_PATCHES))
        self.val_glu = list(range(0, N_GLU_PATCHES, N_GLU_PATCHES // N_GLU_TARGETS))[:N_GLU_TARGETS]

    def _step(self, batch, ctx, tgt, glu_tgt):
        glucose, gluco_img = batch
        dev = glucose.device
        return x_forward_loss(
            self.cgm_encoder, self.cgm_encoder_ema, self.cgm_predictor,
            self.glu_encoder, self.glu_predictor, glucose, gluco_img,
            torch.tensor(ctx, device=dev), torch.tensor(tgt, device=dev),
            torch.tensor(glu_tgt, device=dev), GLUCO_LOSS_WEIGHT, SIGREG_WEIGHT)

    def training_step(self, batch, _):
        ctx, tgt = sample_block_mask(N_CGM_PATCHES, N_CGM_TARGETS, 2, 4, self.rng)
        glu_tgt = sorted(self.rng.sample(range(N_GLU_PATCHES), N_GLU_TARGETS))
        total, cgm, gluco, reg = self._step(batch, ctx, tgt, glu_tgt)
        self.log_dict({"train_total": total, "train_cgm": cgm,
                       "train_gluco": gluco, "train_sigreg": reg}, prog_bar=True)
        return total

    def on_train_batch_end(self, *_):
        m = momentum_at(self.global_step, self.trainer.estimated_stepping_batches, EMA_BASE)
        ema_update(self.cgm_encoder_ema, self.cgm_encoder, m)

    def validation_step(self, batch, _):
        total, cgm, gluco, reg = self._step(batch, self.val_ctx, self.val_tgt, self.val_glu)
        with torch.no_grad():
            cgm_std, cgm_rank = collapse_metrics(self.cgm_encoder(batch[0]))
            glu_std, glu_rank = collapse_metrics(self.glu_encoder(batch[1]))
        self.log_dict({"val_loss": total, "val_cgm": cgm, "val_gluco": gluco,
                       "val_sigreg": reg,
                       "cgm_latent_std": cgm_std, "cgm_eff_rank": cgm_rank,
                       "glu_latent_std": glu_std, "glu_eff_rank": glu_rank}, prog_bar=True)
        return total

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=LR, weight_decay=0.04)

torch.set_float32_matmul_precision("high")   # use tensor cores if the GPU has them

pl.seed_everything(SEED)
train_loader = DataLoader(PairedDataset(train_w, train_i), batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True, num_workers=2, persistent_workers=True)
val_loader   = DataLoader(PairedDataset(val_w, val_i), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, persistent_workers=True)

ckpt   = pl.callbacks.ModelCheckpoint(
    dirpath=f"{DRIVE_DIR}/checkpoints", filename="xjepa-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=-1, save_last=True)  # keep every epoch; pick from metrics.csv
logger = pl.loggers.CSVLogger(save_dir=DRIVE_DIR, name="logs")

trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="auto", devices=1,
                     callbacks=[ckpt], logger=logger, log_every_n_steps=1)
trainer.fit(XJepa(), train_loader, val_loader)

print("best checkpoint:", ckpt.best_model_path)
print("logs (metrics.csv):", logger.log_dir)

## Plot logged metrics

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

csvs = sorted(glob.glob(f"{DRIVE_DIR}/logs/version_*/metrics.csv"), key=os.path.getmtime)
df = pd.read_csv(csvs[-1])   # newest run
print("plotting:", csvs[-1], "|", len(df), "rows")

groups = [
    ("total loss",              ["train_total", "val_loss"]),
    ("cgm loss (EMA target)",   ["train_cgm", "val_cgm"]),
    ("gluco loss",              ["train_gluco", "val_gluco"]),
    ("sigreg",                  ["train_sigreg", "val_sigreg"]),
    ("cgm_latent_std",          ["cgm_latent_std"]),
    ("cgm_eff_rank / 96",       ["cgm_eff_rank"]),
    ("glu_latent_std",          ["glu_latent_std"]),
    ("glu_eff_rank / 96",       ["glu_eff_rank"]),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (title, cols) in zip(axes.ravel(), groups):
    plotted = False
    for c in cols:
        if c in df.columns and df[c].notna().any():
            s = df[["step", c]].dropna()
            ax.plot(s["step"], s[c], marker=".", ms=3, label=c)
            plotted = True
    ax.set_title(title); ax.set_xlabel("step"); ax.grid(alpha=0.3)
    if plotted:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "not logged in this run", ha="center", va="center",
                transform=ax.transAxes, color="gray")
fig.tight_layout()
plt.show()

## Encoder diagnostics (repo's plot_encoder_diagnostics)

4 panels per encoder: activation histogram, sorted per-dimension std, PCA of the
pooled embeddings coloured by window shape, and cumulative explained variance —
the visual form of `eff_rank`. A Gaussian-looking histogram + flat std bars +
a slow-rising variance curve = healthy/isotropic; a spike + bars to zero + a
curve that saturates in 2-3 components = collapse.

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def window_trend(w):
    w = np.asarray(w, np.float64)
    return (w[:, -1] - w[:, 0]) / np.maximum(w.std(1), 1e-6)

def eff_rank_std(lat):
    z = lat.reshape(-1, lat.shape[-1]).astype(np.float64)
    std = z.std(0).mean()
    zc = z - z.mean(0)
    ev = np.linalg.eigvalsh(zc.T @ zc / max(len(z) - 1, 1)).clip(0)
    return std, (ev.sum() ** 2 / (ev ** 2).sum())

def encoder_diagnostics(latents, colors, title):
    arr = np.asarray(latents, np.float64)
    pooled = arr.mean(1) if arr.ndim == 3 else arr        # (N, E)
    tokens = arr.reshape(-1, arr.shape[-1])               # (N*P, E)
    per_dim_std = tokens.std(0)
    embed_dim = pooled.shape[1]
    pca = PCA(n_components=min(embed_dim, pooled.shape[0], 20)).fit(pooled)
    proj, var = pca.transform(pooled), pca.explained_variance_ratio_

    fig, ax = plt.subplots(2, 2, figsize=(12, 9)); fig.suptitle(title, fontsize=13)
    ax[0, 0].hist(tokens.ravel(), bins=80, color="#4C72B0")
    ax[0, 0].set_title("Latent activation distribution (all patches)"); ax[0, 0].set_xlabel("activation"); ax[0, 0].set_ylabel("count")
    ax[0, 1].bar(np.arange(embed_dim), np.sort(per_dim_std)[::-1], color="#DD8452", width=1.0)
    ax[0, 1].axhline(per_dim_std.mean(), color="k", ls="--", lw=1, label=f"mean = {per_dim_std.mean():.3f}  (= latent_std)")
    ax[0, 1].set_title("Per-dimension std across patches (sorted)"); ax[0, 1].set_xlabel("dimension (sorted)"); ax[0, 1].set_ylabel("std"); ax[0, 1].set_ylim(bottom=0); ax[0, 1].legend(fontsize=8)
    sc = ax[1, 0].scatter(proj[:, 0], proj[:, 1], c=colors, cmap="viridis", s=10, alpha=0.75)
    fig.colorbar(sc, ax=ax[1, 0], label="window trend  (last - first) / std")
    ax[1, 0].set_title("PCA of pooled embeddings"); ax[1, 0].set_xlabel(f"PC1 ({var[0]*100:.1f}% var)"); ax[1, 0].set_ylabel(f"PC2 ({var[1]*100:.1f}% var)")
    cum = np.cumsum(var)
    ax[1, 1].plot(np.arange(1, len(cum) + 1), cum, marker="o", ms=3, color="#C44E52"); ax[1, 1].axhline(0.95, color="k", ls="--", lw=1, label="95%")
    ax[1, 1].set_title("Cumulative explained variance"); ax[1, 1].set_xlabel("component"); ax[1, 1].set_ylabel("cumulative ratio"); ax[1, 1].set_ylim(0, 1.02); ax[1, 1].legend(fontsize=8)
    fig.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

def param_distribution(enc, title):
    """Weight distribution of an encoder: histogram of all weights + per-tensor std."""
    named = [(k, v.detach().cpu().numpy().ravel()) for k, v in enc.state_dict().items() if v.ndim >= 1]
    allw = np.concatenate([v for _, v in named])
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.5)); fig.suptitle(title, fontsize=13)
    ax[0].hist(allw, bins=100, color="#4C72B0")
    ax[0].set_title(f"All weights  (n={allw.size}, mean={allw.mean():.3f}, std={allw.std():.3f})")
    ax[0].set_xlabel("value"); ax[0].set_ylabel("count")
    ax[1].bar(range(len(named)), [v.std() for _, v in named], color="#DD8452")
    ax[1].set_xticks(range(len(named)))
    ax[1].set_xticklabels([k for k, _ in named], rotation=90, fontsize=6)
    ax[1].set_title("Per-tensor weight std"); ax[1].set_ylabel("std")
    fig.tight_layout(rect=(0, 0, 1, 0.9)); plt.show()

# Load encoders from a saved checkpoint (any epoch / any variant; no live model needed).
CKPT = f"{DRIVE_DIR}/checkpoints/xjepa-epoch=05-val_loss=0.6584.ckpt"
print("diagnostics from:", CKPT)
sd = torch.load(CKPT, map_location="cpu", weights_only=False)["state_dict"]
cgm_enc = JepaEncoder(WINDOW, PATCH, EMBED_DIM, 3, 6)
glu_enc = GlucoEncoder(GRIDSIZE, 8, 3, EMBED_DIM)
cgm_enc.load_state_dict({k[len("cgm_encoder."):]: v for k, v in sd.items() if k.startswith("cgm_encoder.")})
glu_enc.load_state_dict({k[len("glu_encoder."):]: v for k, v in sd.items() if k.startswith("glu_encoder.")})
cgm_enc.eval(); glu_enc.eval()

# 1) activation diagnostics for both encoders
n = min(512, len(train_w))
gsample = torch.from_numpy(train_w[:n]).float()
isample = torch.from_numpy(train_i[:n]).float()
colors = window_trend(train_w[:n])
with torch.no_grad():
    cgm_lat = cgm_enc(gsample).numpy()
    glu_lat = glu_enc(isample).numpy()
for name, lat in [("CGM encoder", cgm_lat), ("Glucodensity encoder", glu_lat)]:
    std, rank = eff_rank_std(lat)
    encoder_diagnostics(lat, colors, f"{name} - latent_std={std:.4f} - eff_rank={rank:.1f}/{lat.shape[-1]}")

# 2) CGM encoder weight distribution
param_distribution(cgm_enc, "CGM encoder - parameter (weight) distribution")